## Analysis: 

**Purpose**: Identify a suitable risk proxy for construction projects

### Question / Hypothesis

**Origin**: Based on the goal of identifying a suitable risk proxy for construction projects 

**Question**: What is the impact of each project and what are the features needed to determine project duration and potential for extension?

**Assumptions**:
- **Data**: Miami-Dade FDOT Work Program (post-ingestion: LOC_ERROR == "NO ERROR")
- **Fiscal years**: [e.g., 2024–2029 or subset]
- **Features under test**: [e.g., Shape__Length, WPPHAZGP, spatial density]
- **Other**: [Any filtering or subsetting]

### Data Loading

In [109]:
import geopandas as gpd
import pandas as pd
import numpy as np
from pathlib import Path

# Import each layer's data
BASE_PATH = Path.cwd().parent / "src" / "data" / "processed"
LAYERS = ['admin', 'construction', 'mot', 'planning']
data_files = {}

for layer in LAYERS:
    gdf = gpd.read_file(BASE_PATH / f"fdot_work_program_{layer}.gpkg")
    data_files[layer] = gdf

construction_gdf = data_files["construction"]


## Data Filtering & Preparation

Apply any subsetting (e.g., fiscal year, phase type). Check schema, missingness, duplicates, geometry. See [Data Dictionary](../README.md#data-dictionary).

In [110]:
# Count total rows in construction_gdf
print(f"Total rows in construction_gdf: {construction_gdf.shape}")

# Display unique fiscal years
print(construction_gdf["FISCALYR"].unique())


Total rows in construction_gdf: (6942, 28)
[2025 2023 2026 2024 2027 2028 2030 2029]


In [151]:
work_mix = construction_gdf[["WPWKMIX", "WPWKMIXN"]].groupby("WPWKMIX")
# Create a table of work mix codes and their assc. names
work_mix_ref = (
    construction_gdf[["WPWKMIX", "WPWKMIXN"]]
    .drop_duplicates(subset=["WPWKMIX", "WPWKMIXN"])
    .sort_values(["WPWKMIX", "WPWKMIXN"])
)
display(work_mix_ref)


,WPWKMIX,WPWKMIXN
7,0002,NEW ROAD CONSTRUCTIO
973,0005,FLEXIBLE PAVEMENT RE
1811,0010,TRAFFIC OPS IMPROVEM
800,0012,RESURFACING
1826,0015,RESURFACING - RIDE O
714,0022,BRIDGE REPLACEMENT
0,0023,BRIDGE-REPLACE AND A
2133,0024,BRIDGE-REPAIR/REHABI
1990,0106,BIKE PATH/TRAIL
4999,0107,BIKE LANE/SIDEWALK


In [156]:
# Display all unique phase groups
phase_groups = construction_gdf["WPPHAZTP"].unique()
phase_groups_table = pd.DataFrame(phase_groups, columns=["WPPHAZTP"])
phase_groups_table.sort_values(by="WPPHAZTP", ascending=True, inplace=True)
display(phase_groups_table)

# Group by phase type and get the mode of the other columns
construction_gdf.groupby("WPPHAZTP")[[
    "WPITSTNM",
    "WPWKMIXN",
    "LOCALFULL"
]].agg(lambda x: x.mode().iloc[0] if not x.mode().empty else None)

,WPPHAZTP
0,2
4,3
6,4
1,6
2,7
5,8
3,A


,WPITSTNM,WPWKMIXN,LOCALFULL
WPPHAZTP,,,
2,UNDER CONSTRUCTION,RESURFACING,SR 836/I-395 FROM WEST OF I-95 TO MACARTHUR CSWY BRIDGE
3,UNDER CONSTRUCTION,ADD LANES & RECONSTR,SR 826/PALMETTO EXPRESSWAY FROM SOUTH OF NW 36 ST TO N OF NW 154 ST
4,CONTRACT EXECUTED,PEDESTRIAN SAFETY IM,CITY OF MIAMI BEACH- SOUTH BEACH PEDESTRIANPRIORITY ZONE
6,UNDER CONSTRUCTION,ADD LANES & RECONSTR,GOLDEN GLADES INTERCHANGE IMPROVEMENTS - SPUR
7,UNDER CONSTRUCTION,ADD LANES & RECONSTR,SR 7 AND SR 9/GOLDEN GLADES INTERCHANGE VARIOUS RAMP IMPROVEMENTS
8,PRE-CONST.UNDERWAY,BIKE PATH/TRAIL,CITYWIDE SIDEWALK CONNECTIVITY PROJECT - PHASE I
A,CONST.COMPLETE,RESURFACING,GOLDEN GLADES INTERCHANGE IMPROVEMENTS (MAINLINE SPUR MP 0X)


Based on the phase type data, we are able to build a legend for each type based on the char code

In [132]:
phase_type_legend = {
    "2": "Active Construction",
    "3": "Active Construction",
    "4": "Contract Executed",
    "6": "Active Construction",
    "7": "Active Construction",
    "8": "Pre-Construction",
    "A": "Construction Completed",
}

# Adding a new column based on the phase type legend
construction_gdf["WPPHAZTP_DESC"] = construction_gdf["WPPHAZTP"].map(phase_type_legend)

# Display the new column
construction_gdf[["WPPHAZTP", "WPPHAZTP_DESC"]].head()

,WPPHAZTP,WPPHAZTP_DESC
0,2,Active Construction
1,2,Active Construction
2,2,Active Construction
3,2,Active Construction
4,2,Active Construction


In [163]:
# Assign weights to each phase type
phase_weights = {
    "Pre-Construction": 0.25,
    "Contract Executed": 0.5,
    "Active Construction": 1.0,
    "Construction Completed": 0.1
}

# Add phase weight to construction_gdf
construction_gdf["PHASE_WEIGHT"] = construction_gdf["WPPHAZTP_DESC"].map(phase_weights)

# Display the first few and last few rows of the DataFrame
display(construction_gdf[["WPPHAZTP_DESC", "PHASE_WEIGHT"]].head())
display(construction_gdf[["WPPHAZTP_DESC", "PHASE_WEIGHT"]].tail())


,WPPHAZTP_DESC,PHASE_WEIGHT
4716,Construction Completed,0.1
4717,Construction Completed,0.1
4720,Construction Completed,0.1
4662,Construction Completed,0.1
4747,Construction Completed,0.1


,WPPHAZTP_DESC,PHASE_WEIGHT
86,Active Construction,1.0
80,Active Construction,1.0
81,Active Construction,1.0
66,Active Construction,1.0
67,Active Construction,1.0


## Calculations

Compute metrics for risk-proxy candidate features.
Align with impact_score: `segment_length`, `phase_weight`

In [142]:
# Risk Proxy Calculation
construction_gdf["risk_proxy"] = construction_gdf["Shape__Length"] * construction_gdf["PHASE_WEIGHT"]

# Display the new column
construction_gdf[["WPPHAZTP_DESC", "WPWKMIXN", "Shape__Length", "PHASE_WEIGHT", "risk_proxy"]].head()

# construction_gdf["PHASE_WEIGHT"].value_counts()

,WPPHAZTP_DESC,WPWKMIXN,Shape__Length,PHASE_WEIGHT,risk_proxy
0,Active Construction,BRIDGE-REPLACE AND A,115.922570,1.0,115.922570
1,Active Construction,BRIDGE-REPLACE AND A,279.354608,1.0,279.354608
2,Active Construction,BRIDGE-REPLACE AND A,170.858574,1.0,170.858574
3,Active Construction,BRIDGE-REPLACE AND A,2177.798387,1.0,2177.798387
4,Active Construction,BRIDGE-REPLACE AND A,115.922570,1.0,115.922570


### Risk Proxy Construction

We define risk as:

`risk_proxy = segment_length * phase_weight`

Where:
- `segment_length` represents the spatial expanse of each project
- `phase_weight` reflects the expected level of disruption based on project stage (under construction, completed, etc.)

Analysis shows that while most projects are actively under construction (5892/6942 rows), a non-trivial total are in one of the other phase categories (pre-construction, contract executed, completed)
- Incorporating phase weights allow us to better capture the relative impact of these stages.

## Test

Evaluate the hypothesis. EDA checklist (README): feature variance, non-pathological distributions, correlated structure, interpretability.

### Variance & distribution

Check variance (avoid near-zero), distribution shape (no single-feature dominance).

### Correlations (multicollinearity)

Flag strong correlations (|r| > 0.8) between candidate features.

### Geospatial (optional)

Map sample, bounds, or density visualization.

## Conclusion

Summarize findings and whether the feature(s) are viable for the risk-proxy model.

**Variance**: [Sufficient / near-zero / skewed?]

**Distribution**: [Non-pathological? Any single-feature dominance?]

**Correlations**: [Multicollinearity concerns?]

**Feature viability**: [Supported / not supported / inconclusive] for use in impact_score

## Next Steps

- [Follow-up hypothesis to test]
- [Additional feature(s) to evaluate]
- [Refinements to modeling pipeline]